# TrOCR fine-tuning on RUKOPYS (Mavka team)

Runtime → Change runtime type → **T4 GPU** (or better).

**Where things live**
- Dataset (19.3 GB) → Colab's local disk `/content/rukopys`. It is **not** copied to Drive.
- Line crops → local disk, plus one archive copied to Drive so a new session can skip the download.
- Checkpoints and final models → Drive, so training survives disconnects.

**Order:** run cells 1–5 once. Then run the training cell with `STAGE="A"` (silver + gold), then with `STAGE="B"` (gold only).
If Colab disconnects, run cells 1–3 again (they restore crops from Drive), then rerun training: it resumes from the last checkpoint.

## 1. GPU, disk, packages

In [7]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1
# Pinned to transformers 4.x because this notebook uses Seq2SeqTrainer and VisionEncoderDecoderModel.
!pip -q install "transformers>=4.45,<5" "accelerate>=0.34" evaluate jiwer huggingface_hub

name, memory.total [MiB]
Tesla T4, 15360 MiB
overlay         113G  3.4G  110G   3% /
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 2. Paths and settings

In [8]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DATA  = Path('/content/rukopys')                      # raw dataset, local disk only
CROPS = Path('/content/crops')                        # line crops, local disk
OUT   = Path('/content/drive/MyDrive/rukopys_trocr')  # checkpoints + models + crop archive
OUT.mkdir(parents=True, exist_ok=True)
CROPS_TAR = OUT / 'crops.tar'

BASE_MODEL  = 'kazars24/trocr-base-handwritten-ru'    # or 'Kansallisarkisto/cyrillic-htr-model'
USE_SILVER  = False       # True only if your crops.tar includes silver (made without --no-silver)
SILVER_LIMIT = None       # e.g. 100_000 to shorten stage A on a free T4
VAL_FRAC, SEED = 0.2, 42
MAX_W = 1536              # crops wider than this are downscaled (TrOCR resizes to 384x384 anyway)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Restore crops from Drive (skip download if they exist)

In [9]:
import tarfile, os, shutil
if CROPS_TAR.exists() and CROPS_TAR.stat().st_size < 10_000_000:
    print('Removing an empty/broken crop archive from Drive'); CROPS_TAR.unlink()
gt = CROPS / 'gold_train.jsonl'
if CROPS_TAR.exists() and not (gt.exists() and gt.stat().st_size > 0):
    shutil.rmtree(CROPS, ignore_errors=True)
    print('Restoring crops from Drive...')
    with tarfile.open(CROPS_TAR) as t: t.extractall('/content')
HAVE_CROPS = (CROPS / 'gold_train.jsonl').exists() and (CROPS / 'gold_train.jsonl').stat().st_size > 0
print('crops ready:', HAVE_CROPS)

crops ready: True


## 4. Download the dataset (only if crops are not ready)
If the download asks for authorization, accept the dataset terms on its Hugging Face page,
then add an `HF_TOKEN` secret in Colab (key icon on the left) with notebook access enabled.

In [10]:
from huggingface_hub import snapshot_download
if not HAVE_CROPS:
    try:
        from google.colab import userdata
        os.environ.setdefault('HF_TOKEN', userdata.get('HF_TOKEN') or '')
    except Exception:
        pass
    patterns = ['train/*', 'test/*', '*.csv'] + (['silver/*'] if USE_SILVER else [])
    from huggingface_hub import HfApi
    from collections import Counter
    files = HfApi().list_repo_files('UkrainianCatholicUniversity/rukopys', repo_type='dataset')
    print('files in repo:', len(files))
    print(Counter('/'.join(f.split('/')[:2]) if f.count('/') > 1 else f for f in files).most_common(15))
    print('metadata files:', [f for f in files if f.endswith('.jsonl')])
    snapshot_download('UkrainianCatholicUniversity/rukopys', repo_type='dataset',
                      local_dir=str(DATA), allow_patterns=patterns, max_workers=8)
    found = sorted(str(p.relative_to(DATA)) for p in DATA.rglob('metadata.jsonl'))
    print('downloaded metadata:', found)
    assert (DATA / 'train' / 'metadata.jsonl').exists(), 'train/metadata.jsonl missing - send this output'
!du -sh /content/rukopys/* 2>/dev/null; df -h /content | tail -1

overlay         113G  3.6G  110G   4% /


## 5. Cut line crops + page-level train/val split
Keeps `handwritten`, `printed`, `annotation` with `language=uk` and `legibility=legible`.
`image`/`graph` (no text, not scored for CER) and `formula`/`table` (Qwen branch) are skipped.
Labels are normalized the same way the competition metric normalizes text.

In [11]:
import json, random, re, shutil
from PIL import Image, ImageOps
from tqdm.auto import tqdm

TEXT_TYPES = {'handwritten', 'printed', 'annotation'}
_STRIKE_FIX = re.compile(r'~~(.*?)~~\{(.*?)\}')
_STRIKE = re.compile(r'~~(.*?)~~')
_TABLE = str.maketrans({'—': '-', '–': '-', '«': '"', '»': '"', '“': '"', '”': '"', '„': '"',
                        '‘': "'", '’': "'", 'c': 'с', 'o': 'о', 'p': 'р', 'x': 'х'})
def normalize(t):
    t = _STRIKE_FIX.sub(r'\2', t or '')   # ~~old~~{new} -> new
    t = _STRIKE.sub(r'\1', t)             # ~~text~~ -> text
    return ' '.join(t.translate(_TABLE).split())

def keep(r):
    if r.get('type') not in TEXT_TYPES or r.get('language', 'uk') != 'uk': return False
    if r.get('legibility') == 'illegible': return False
    t = normalize(r.get('text'))
    if not t or t.startswith('[ERROR:'): return False
    x1, y1, x2, y2 = r['bbox']
    return (x2 - x1) >= 4 and (y2 - y1) >= 4

def load_page(path, W, H):
    im = Image.open(path)
    if W and H and im.size != (W, H):          # phone photos: bboxes may refer to EXIF-rotated image
        t = ImageOps.exif_transpose(im)
        if t.size == (W, H): im = t
    return im.convert('RGB')

def crop_split(split):
    meta = DATA / split / 'metadata.jsonl'
    if not meta.exists(): print('no', meta); return []
    out = CROPS / split; out.mkdir(parents=True, exist_ok=True)
    rows = []
    for line in tqdm(meta.read_text(encoding='utf-8').splitlines(), desc=split):
        page = json.loads(line)
        regs = [r for r in (page.get('regions') or []) if keep(r)]
        if not regs: continue
        fn = page.get('file_name') or page.get('image')
        try:
            im = load_page(DATA / split / fn, page.get('image_width'), page.get('image_height'))
        except Exception as e:
            print('skip', fn, e); continue
        stem = Path(fn).stem
        for i, r in enumerate(regs):
            x1, y1, x2, y2 = map(int, r['bbox'])
            c = im.crop((max(0, x1 - 4), max(0, y1 - 4), min(im.width, x2 + 4), min(im.height, y2 + 4)))
            if c.width > MAX_W:
                c = c.resize((MAX_W, max(1, round(c.height * MAX_W / c.width))))
            p = out / f'{stem}_{i}.jpg'
            c.save(p, quality=92)
            rows.append({'path': str(p.relative_to(CROPS)), 'text': normalize(r['text']),
                         'page': stem, 'type': r['type']})
    return rows

def dump(rows, name):
    with open(CROPS / name, 'w', encoding='utf-8') as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print(f'{name}: {len(rows)} crops')

if not HAVE_CROPS:
    CROPS.mkdir(exist_ok=True)
    gold = crop_split('train')
    assert gold, 'No gold crops: the dataset was not downloaded correctly (see cell 4 output).'
    pages = sorted({r['page'] for r in gold}); random.Random(SEED).shuffle(pages)
    val_pages = set(pages[:int(len(pages) * VAL_FRAC)])
    dump([r for r in gold if r['page'] not in val_pages], 'gold_train.jsonl')
    dump([r for r in gold if r['page'] in val_pages], 'gold_val.jsonl')
    dump(crop_split('test'), 'test.jsonl')   # final check only (empty if test has no labels)
    if USE_SILVER:
        dump(crop_split('silver'), 'silver.jsonl')
        shutil.rmtree(DATA / 'silver' / 'images', ignore_errors=True)   # free ~16 GB local disk
    with tarfile.open(CROPS_TAR, 'w') as t:   # backup to Drive (JPEGs: no point compressing)
        t.add(CROPS, arcname='crops')
    print('crop archive on Drive:', round(CROPS_TAR.stat().st_size / 1e9, 2), 'GB')
!df -h /content | tail -1

overlay         113G  3.6G  110G   4% /


In [5]:
import torch, glob, json, numpy as np
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
ck = sorted(glob.glob(str(OUT / 'stageA' / 'checkpoint-*')), key=lambda p: int(p.split('-')[-1]))[-1]
print('checkpoint:', ck, '| GPU capability:', torch.cuda.get_device_capability())
processor = TrOCRProcessor.from_pretrained(BASE_MODEL); tok = processor.tokenizer
model = VisionEncoderDecoderModel.from_pretrained(ck).cuda().eval()
print(model.generation_config)

rows = [json.loads(l) for l in (CROPS / 'gold_train.jsonl').read_text(encoding='utf-8').splitlines()]
lens = np.array([len(tok(r['text']).input_ids) for r in rows])
print('label tokens: median', np.median(lens), '| p95', np.percentile(lens, 95), '| share over 128:', round((lens > 128).mean(), 3))
print('"привіт" ->', tok.tokenize('привіт'))

val = [json.loads(l) for l in (CROPS / 'gold_val.jsonl').read_text(encoding='utf-8').splitlines()][:8]
pix = processor(images=[Image.open(CROPS / r['path']).convert('RGB') for r in val], return_tensors='pt').pixel_values.cuda()
with torch.no_grad():
    a = model.generate(pix, max_length=256)
    b = model.generate(pix, max_length=256, num_beams=1, no_repeat_ngram_size=0, length_penalty=1.0, early_stopping=False)
for r, x, y in zip(val, tok.batch_decode(a, skip_special_tokens=True), tok.batch_decode(b, skip_special_tokens=True)):
    print('GT   :', r['text']); print('AS-IS:', x); print('FIXED:', y); print()

checkpoint: /content/drive/MyDrive/rukopys_trocr/stageA/checkpoint-4000 | GPU capability: (7, 5)


preprocessor_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.90k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/957 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/480 [00:01<?, ?it/s]

GenerationConfig {
  "bos_token_id": 0,
  "decoder_start_token_id": 0,
  "early_stopping": true,
  "eos_token_id": 2,
  "length_penalty": 2.0,
  "max_length": 128,
  "no_repeat_ngram_size": 3,
  "num_beams": 4,
  "pad_token_id": 1,
  "use_cache": false
}

label tokens: median 37.0 | p95 66.0 | share over 128: 0.0
"привіт" -> ['Ð', '¿', 'ÑĢ', 'Ð¸', 'Ð²', 'Ñ', 'ĸ', 'ÑĤ']
GT   : Магія голосу.
AS-IS: �а�� �о�о�
FIXED: �а�� �о�о�

GT   : Спочатку був "брехунець". З нього почи-
AS-IS: ���р��о�ту�в �р�а�н�ь � � � ��о�о� �о�и
FIXED: ������������������р�т �у �р��у�ввввв � � � � � � � � � � � � � � �о�и

GT   : нався день; клац - пронизливе рокотання
AS-IS: нвн �а�н�;�а� ��ои�иер�тнннкн
FIXED: нврн �а�н���нв �лн�у  ��аи�иер�р�оанннннннннннн

GT   : перших тактів "Реве та стогне", а за
AS-IS: �ери ткт� "�в ат т���нк",��
FIXED: �еи ткткт� "�в ат т��а � � � � � � � � � � � � � � � � � � � � � � � � � � � � � �а�р

GT   : а за ним примусово-бадьоре "Говорить Київ!"
AS-IS: нм �рмссо�о -�дооеаое "��рт

In [12]:
import torch, json, evaluate
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
globals().pop('model', None); torch.cuda.empty_cache()
cer = evaluate.load('cer')
CANDIDATES = ['Hukyl/trocr-large-uk-handwritten', 'Hukyl/trocr-large-uk-handwritten-real',
              'Kansallisarkisto/cyrillic-htr-model']
val = [json.loads(l) for l in (CROPS / 'gold_val.jsonl').read_text(encoding='utf-8').splitlines()]
sample = random.Random(0).sample(val, 200)
imgs = [Image.open(CROPS / r['path']).convert('RGB') for r in sample]
refs = [r['text'] for r in sample]
for m in CANDIDATES:
    try:
        proc = TrOCRProcessor.from_pretrained(m); tk = proc.tokenizer
        print(f'\n== {m} | vocab {len(tk)} | "привіт" ->', tk.tokenize('привіт'))
        mdl = VisionEncoderDecoderModel.from_pretrained(m, torch_dtype=torch.float16).cuda().eval()
        preds = []
        with torch.no_grad():
            for i in range(0, len(imgs), 16):
                pix = proc(images=imgs[i:i + 16], return_tensors='pt').pixel_values.cuda().half()
                ids = mdl.generate(pix, max_length=256, num_beams=1, no_repeat_ngram_size=0)
                preds += [normalize(s) for s in tk.batch_decode(ids, skip_special_tokens=True)]
        print('zero-shot CER on 200 val lines:', round(cer.compute(predictions=preds, references=refs), 4))
        for r, p in list(zip(refs, preds))[:3]: print('  GT  :', r); print('  PRED:', p)
        del mdl; torch.cuda.empty_cache()
    except Exception as e:
        print(f'\n== {m}: FAILED -> {type(e).__name__}: {e}')

processor_config.json:   0%|          | 0.00/431 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.23k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/656 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.59M [00:00<?, ?B/s]


== Hukyl/trocr-large-uk-handwritten | vocab 50336 | "привіт" -> ['п', 'р', 'и', 'в', 'і', 'т']


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.23GB            

... (13 outputs left out)

generation_config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

zero-shot CER on 200 val lines: 0.1583
  GT  : способи організації пізнавальної діяльно-
  PRED: Способи оргатрації пізнавальної ділекста
  GT  : Отже, температури понад 42°C становить
  PRED: Отже, температури понад 42"С становить
  GT  : матчах. Аж дивно, яке багатство
  PRED: матчах. Аж дивно, яке багатство


config.json:   0%|          | 0.00/4.74k [00:00<?, ?B/s]


== Kansallisarkisto/cyrillic-htr-model: FAILED -> OSError: Can't load image processor for 'Kansallisarkisto/cyrillic-htr-model'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'Kansallisarkisto/cyrillic-htr-model' is the correct path to a directory containing a preprocessor_config.json file


## 6. Train
Run with `STAGE = "A"` first (silver + gold, 1 epoch), then change to `"B"` and run again (gold only, starts from stage A).
Rerunning the same stage resumes from the last checkpoint on Drive.

In [ ]:
STAGE = 'A'   # 'A' then 'B'

import torch, evaluate
from transformers import (TrOCRProcessor, VisionEncoderDecoderModel, Seq2SeqTrainer,
                          Seq2SeqTrainingArguments, default_data_collator)
from transformers.trainer_utils import get_last_checkpoint

CFG = {
    'A': dict(init=BASE_MODEL, data=['gold_train'] + (['silver'] if USE_SILVER else []),
              epochs=1 if USE_SILVER else 10, lr=4e-5,
              save_steps=2000 if USE_SILVER else 1000),
    'B': dict(init=str(OUT / 'stageA' / 'final'), data=['gold_train'],
              epochs=8, lr=2e-5, save_steps=500),
}[STAGE]
RUN = OUT / f'stage{STAGE}'
MAX_LEN, BS, ACCUM = 128, 8, 2

def read(name):
    p = CROPS / f'{name}.jsonl'
    return [json.loads(l) for l in p.read_text(encoding='utf-8').splitlines()] if p.exists() else []

train_rows = []
for name in CFG['data']:
    rows = read(name)
    if name == 'silver' and SILVER_LIMIT:
        rows = random.Random(SEED).sample(rows, min(SILVER_LIMIT, len(rows)))
    train_rows += rows
val_rows = read('gold_val')
val_small = random.Random(SEED).sample(val_rows, min(1500, len(val_rows)))  # fast eval during training
print(f'stage {STAGE}: train {len(train_rows)}  val {len(val_rows)}')

processor = TrOCRProcessor.from_pretrained(CFG['init'])
model = VisionEncoderDecoderModel.from_pretrained(CFG['init'])
tok = processor.tokenizer
for ch in "іїєґІЇЄҐ'ʼ":
    if tok.unk_token_id is not None and tok.unk_token_id in tok(ch, add_special_tokens=False).input_ids:
        print('WARNING: tokenizer has no token for', repr(ch))

start = tok.cls_token_id if tok.cls_token_id is not None else tok.bos_token_id
eos = tok.sep_token_id if tok.sep_token_id is not None else tok.eos_token_id
for c in (model.config, model.generation_config):
    c.decoder_start_token_id, c.pad_token_id, c.eos_token_id = start, tok.pad_token_id, eos
model.generation_config.max_length = MAX_LEN

class Crops(torch.utils.data.Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        im = Image.open(CROPS / r['path']).convert('RGB')
        pix = processor(images=im, return_tensors='pt').pixel_values[0]
        ids = tok(r['text'], max_length=MAX_LEN, truncation=True, padding='max_length').input_ids
        return {'pixel_values': pix,
                'labels': torch.tensor([t if t != tok.pad_token_id else -100 for t in ids])}

cer_metric = evaluate.load('cer')
def compute_metrics(pred):
    p, l = pred.predictions, pred.label_ids
    p[p == -100] = tok.pad_token_id; l[l == -100] = tok.pad_token_id
    preds = [normalize(s) for s in tok.batch_decode(p, skip_special_tokens=True)]
    refs = [normalize(s) for s in tok.batch_decode(l, skip_special_tokens=True)]
    return {'cer': cer_metric.compute(predictions=preds, references=refs)}

args = Seq2SeqTrainingArguments(
    output_dir=str(RUN), num_train_epochs=CFG['epochs'], learning_rate=CFG['lr'],
    per_device_train_batch_size=BS, gradient_accumulation_steps=ACCUM, per_device_eval_batch_size=BS * 2,
    warmup_ratio=0.05, weight_decay=0.01, lr_scheduler_type='cosine', max_grad_norm=1.0,
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    eval_strategy='steps', eval_steps=CFG['save_steps'],
    save_strategy='steps', save_steps=CFG['save_steps'], save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model='cer', greater_is_better=False,
    predict_with_generate=True, generation_num_beams=1, generation_max_length=MAX_LEN,
    dataloader_num_workers=2, logging_steps=100, report_to='none',
)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=Crops(train_rows),
                         eval_dataset=Crops(val_small), data_collator=default_data_collator,
                         compute_metrics=compute_metrics)
last = get_last_checkpoint(str(RUN)) if RUN.exists() else None
print('resuming from', last)
trainer.train(resume_from_checkpoint=last)

final = RUN / 'final'
trainer.save_model(str(final)); processor.save_pretrained(str(final))
print('saved', final)

stage A: train 17779  val 4333


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


resuming from None


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss,Cer
1000,0.994800,0.893703,0.898067
2000,0.705300,0.661982,0.830437
3000,0.509200,0.584329,0.843232
4000,0.376400,0.546229,0.873179


The following generation flags are not valid and may be ignored: ['early_stopping', 'length_penalty']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
